## 1) Instalación de dependencias
Instala `mlxtend`, `pandas` y `seaborn` si no están presentes. Ejecutar una vez en entornos limpios.

In [18]:
# Instala dependencias necesarias (ejecutar una vez)
import sys
import subprocess
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
try:
    import mlxtend
except Exception:
    pip_install('mlxtend')
try:
    import pandas as pd
except Exception:
    pip_install('pandas')
try:
    import seaborn as sns
except Exception:
    pip_install('seaborn')
print('Dependencias listas')

Dependencias listas


## 2) Imports y configuración
Carga de librerías y configuración de visualización.

In [19]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
plt.rcParams['figure.figsize'] = (8,5)
sns.set(style='whitegrid')
print('Entorno importado')

Entorno importado


## 3) Cargar dataset
Busca un CSV con 'diabetes' en el nombre y lo carga en `df`.

In [20]:
candidates = glob.glob('*diabetes*.csv') + glob.glob('*Diabetes*.csv')
candidates = sorted(list(set(candidates)))
if not candidates:
    raise FileNotFoundError('No se encontró un CSV con "diabetes" en el nombre en el directorio actual. Coloque el archivo y vuelva a ejecutar.')
path = candidates[0]
print('Leyendo:', path)
df = pd.read_csv(path)
df.shape

Leyendo: diabetes_BRFSS2015.csv


(253680, 22)

## 4) Vista rápida y tipos
Muestra `head()`, `info()` y `describe()` para inspeccionar el dataset.

In [21]:
display(df.head())
display(df.info())
df.describe(include='all')

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


<class 'pandas.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  MentHlth   

None

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,...,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000
mean,0.296921,0.429001,0.424121,0.962670,28.382364,0.443169,0.040571,0.094186,0.756544,0.634256,...,0.951053,0.084177,2.511392,3.184772,4.242081,0.168224,0.440342,8.032119,5.050434,6.053875
std,0.698160,0.494934,0.494210,0.189571,6.608694,0.496761,0.197294,0.292087,0.429169,0.481639,...,0.215759,0.277654,1.068477,7.412847,8.717951,0.374066,0.496429,3.054220,0.985774,2.071148
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,5.000000
50%,0.000000,0.000000,0.000000,1.000000,27.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,7.000000
75%,0.000000,1.000000,1.000000,1.000000,31.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,2.000000,3.000000,0.000000,1.000000,10.000000,6.000000,8.000000
max,2.000000,1.000000,1.000000,1.000000,98.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


## 5) Identificar columnas binarias
Detecta columnas con valores 0/1 y muestra las más prevalentes.

In [22]:
binary_cols = [c for c in df.columns if set(df[c].dropna().unique()).issubset({0,1})]
print('Columnas binarias detectadas (cantidad):', len(binary_cols))
display(pd.Series({c: df[c].sum() for c in binary_cols}).sort_values(ascending=False).head(20))

Columnas binarias detectadas (cantidad): 14


CholCheck               244210.0
AnyHealthcare           241263.0
Veggies                 205841.0
PhysActivity            191920.0
Fruits                  160898.0
Smoker                  112423.0
Sex                     111706.0
HighBP                  108829.0
HighChol                107591.0
DiffWalk                 42675.0
HeartDiseaseorAttack     23893.0
NoDocbcCost              21354.0
HvyAlcoholConsump        14256.0
Stroke                   10292.0
dtype: float64

## 6) Detección y eliminación de duplicados
Cuenta y elimina duplicados si existen.

In [23]:
dups = df.duplicated().sum()
print('Registros duplicados encontrados:', dups)
if dups>0:
    df = df.drop_duplicates().reset_index(drop=True)
    print('Duplicados eliminados. Nuevo shape:', df.shape)
else:
    print('No hay duplicados')

Registros duplicados encontrados: 23899
Duplicados eliminados. Nuevo shape: (229781, 22)


## 7) Manejo de valores nulos
Elimina columnas que contengan cualquier null para evitar reglas espurias.

In [24]:
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts>0].sort_values(ascending=False)
print('Columnas con nulls (a eliminar):', len(null_cols))
if len(null_cols)>0:
    display(null_cols)
    df = df.drop(columns=null_cols.index)
    print('Columnas con null eliminadas. Nuevo shape:', df.shape)
else:
    print('No se encontraron valores nulos por columna')

Columnas con nulls (a eliminar): 0
No se encontraron valores nulos por columna


## 8) Guardar dataset crudo
Guarda `data_raw.csv` tras la limpieza de duplicados y nulos.

In [25]:
raw_path = 'data_raw.csv'
df.to_csv(raw_path, index=False)
print('Guardado dataset crudo en', raw_path)

Guardado dataset crudo en data_raw.csv


## 9) Discretización
Discretiza `BMI` (si existe) y `Age` en cuartiles; aplica `qcut` a otras variables numéricas no binarias.

In [26]:
df_disc = df.copy()
if 'BMI' in df_disc.columns:
    bins = [0, 18.5, 25, 30, np.inf]
    labels = ['Underweight','Normal','Overweight','Obese']
    df_disc['BMI_cat'] = pd.cut(df_disc['BMI'].astype(float), bins=bins, labels=labels)
    df_disc = df_disc.drop(columns=['BMI'])
    print('Discretizada columna BMI -> BMI_cat')
else:
    print('No se detectó columna BMI')
if 'Age' in df_disc.columns or 'age' in df_disc.columns:
    age_col = 'Age' if 'Age' in df_disc.columns else 'age'
    try:
        df_disc['Age_cat'] = pd.qcut(df_disc[age_col].astype(float), q=4, labels=['A1','A2','A3','A4'])
        df_disc = df_disc.drop(columns=[age_col])
        print(f'Discretizada {age_col} -> Age_cat')
    except Exception as e:
        print('No se pudo discretizar age:', e)
for c in df_disc.select_dtypes(include=['float','int']).columns:
    if set(df_disc[c].dropna().unique()).issubset({0,1}):
        continue
    try:
        df_disc[c+'_cat'] = pd.qcut(df_disc[c], q=4, duplicates='drop')
        df_disc = df_disc.drop(columns=[c])
        print('Discretizada', c)
    except Exception:
        pass
for c in df_disc.select_dtypes(include=['category','object']).columns:
    df_disc[c] = df_disc[c].astype(str)
disc_path = 'data_discretized.csv'
df_disc.to_csv(disc_path, index=False)
print('Guardado dataset discretizado en', disc_path)

Discretizada columna BMI -> BMI_cat
Discretizada Age -> Age_cat
Discretizada Diabetes_012
Discretizada GenHlth
Discretizada MentHlth
Discretizada PhysHlth
Discretizada Education
Discretizada Income
Guardado dataset discretizado en data_discretized.csv


## 10) Preparar datos para reglas de asociación
Crea una matriz one-hot a partir de `df_disc`.

In [27]:
def to_onehot_for_association(df_in):
    df_local = df_in.copy()
    bin_cols = [c for c in df_local.columns if set(df_local[c].dropna().unique()).issubset({0, 1})]
    df_bin = df_local[bin_cols].astype(bool) if bin_cols else pd.DataFrame(index=df_local.index)
    cat_cols = [c for c in df_local.columns if c not in bin_cols]
    df_cat = pd.get_dummies(df_local[cat_cols].astype(str), prefix_sep='=') if cat_cols else pd.DataFrame(index=df_local.index)
    return pd.concat([df_bin, df_cat], axis=1).astype(bool)

df_onehot = to_onehot_for_association(df_disc)
print('One-hot shape:', df_onehot.shape)
display(df_onehot.head())

One-hot shape: (229781, 36)


,HighBP,HighChol,CholCheck,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,...,"MentHlth_cat=(-0.001, 2.0]","MentHlth_cat=(2.0, 30.0]","PhysHlth_cat=(-0.001, 4.0]","PhysHlth_cat=(4.0, 30.0]","Education_cat=(0.999, 4.0]","Education_cat=(4.0, 5.0]","Education_cat=(5.0, 6.0]","Income_cat=(0.999, 5.0]","Income_cat=(5.0, 6.0]","Income_cat=(6.0, 8.0]"
0,True,True,True,True,False,False,False,False,True,False,...,False,True,False,True,True,False,False,True,False,False
1,False,False,False,True,False,False,True,False,False,False,...,True,False,True,False,False,False,True,True,False,False
2,True,True,True,False,False,False,False,True,False,False,...,False,True,False,True,True,False,False,False,False,True
3,True,False,True,False,False,False,True,True,True,False,...,True,False,True,False,True,False,False,False,True,False
4,True,True,True,False,False,False,True,True,True,False,...,False,True,True,False,False,True,False,True,False,False


## 11) Minería de reglas (Apriori)
Calcula itemsets frecuentes y genera reglas filtrando por `min_confidence` y `lift>1`. Ajustar `min_support` según recursos.

In [ ]:
min_support = 0.01
min_confidence = 0.6
freq_itemsets = fpgrowth(df_onehot, min_support=min_support, use_colnames=True)
print('Itemsets frecuentes:', len(freq_itemsets))
if len(freq_itemsets):
    rules = association_rules(freq_itemsets, metric='confidence', min_threshold=min_confidence)
    rules = rules[rules['lift'] > 1].sort_values(['confidence', 'lift'], ascending=False)
    display(rules.head(20))
    rules.to_csv('association_rules.csv', index=False)
    print('Guardado: association_rules.csv')
else:
    print('No se encontraron itemsets con el soporte mínimo dado')

Itemsets frecuentes: 527389


## Notas finales
- Se han creado `data_raw.csv` y `data_discretized.csv` tras ejecutar el notebook.
- Ajuste `min_support`/`min_confidence` según objetivos y recursos.